# 추천시스템 - 협업 필터링 (Collaborative Filtering)

사용자들의 평점 데이터를 기반으로, 취향이 비슷한 사용자가 좋아한 아이템을 추천하는 방식

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## 1. 샘플 데이터 생성

사용자 5명이 영화 6편에 매긴 평점 (NaN = 안 봄)

In [4]:
ratings =

ratings

<function pandas.read_csv(filepath_or_buffer: 'FilePath | ReadCsvBuffer[bytes] | ReadCsvBuffer[str]', *, sep: 'str | None | lib.NoDefault' = <no_default>, delimiter: 'str | None | lib.NoDefault' = None, header: "int | Sequence[int] | None | Literal['infer']" = 'infer', names: 'Sequence[Hashable] | None | lib.NoDefault' = <no_default>, index_col: 'IndexLabel | Literal[False] | None' = None, usecols: 'UsecolsArgType' = None, dtype: 'DtypeArg | None' = None, engine: 'CSVEngine | None' = None, converters: 'Mapping[HashableT, Callable] | None' = None, true_values: 'list | None' = None, false_values: 'list | None' = None, skipinitialspace: 'bool' = False, skiprows: 'list[int] | int | Callable[[Hashable], bool] | None' = None, skipfooter: 'int' = 0, nrows: 'int | None' = None, na_values: 'Hashable | Iterable[Hashable] | Mapping[Hashable, Iterable[Hashable]] | None' = None, keep_default_na: 'bool' = True, na_filter: 'bool' = True, skip_blank_lines: 'bool' = True, parse_dates: 'bool | Sequence[

## 2. 결측값 채우기

코사인 유사도 계산을 위해 NaN을 0으로 채움 (안 본 영화 = 0점 취급)

In [5]:
ratings_filled = ratings.fillna(0)
ratings_filled = ratings.apply(lambda row:row.fillna(row.mean()), axis=1)
ratings_filled
ratings_filled

AttributeError: 'function' object has no attribute 'fillna'

## 3. 사용자 간 유사도 계산 (코사인 유사도)

각 사용자의 평점 벡터를 비교해서 취향이 얼마나 비슷한지 측정 (-1 ~ 1, 1에 가까울수록 비슷)

In [6]:
user_sim = cosine_similarity(ratings_filled)
user_sim_df = pd.DataFrame(user_sim, index=ratings.index, columns=ratings.index)
user_sim_df.round(2)

,유저A,유저B,유저C,유저D,유저E
유저A,1.00,0.96,0.82,0.85,0.76
유저B,0.96,1.00,0.86,0.93,0.81
유저C,0.82,0.86,1.00,0.90,0.98
유저D,0.85,0.93,0.90,1.00,0.90
유저E,0.76,0.81,0.98,0.90,1.00


## 4. 추천 함수

특정 사용자가 아직 안 본 영화 중, 비슷한 사용자들의 평점을 유사도 가중 평균으로 계산하여 추천

In [1]:
def recommend(user, ratings, user_sim_df, top_n=3):
    # 해당 사용자가 안 본 영화 목록
    unseen = ratings.loc[user][ratings.loc[user].isna()].index

    scores = {}
    for movie in unseen:
        # 이 영화를 본 다른 사용자들
        watched = ratings[movie].dropna().index
        watched = watched[watched != user]

        if len(watched) == 0:
            continue

        # 유사도 가중 평균: sum(유사도 * 평점) / sum(유사도)
        sim = user_sim_df.loc[user, watched]
        rate = ratings.loc[watched, movie]
        scores[movie] = np.dot(sim, rate) / sim.sum()

    result = pd.Series(scores).sort_values(ascending=False)
    return result.head(top_n)

## 5. 추천 결과 확인

In [2]:
# 유저B에게 추천
print('유저B가 안 본 영화:', list(ratings.loc['유저B'][ratings.loc['유저B'].isna()].index))
print()
print('추천 결과 (예상 평점):')
recommend('유저B', ratings, user_sim_df)

NameError: name 'ratings' is not defined

In [ ]:
# 유저D에게 추천
print('유저D가 안 본 영화:', list(ratings.loc['유저D'][ratings.loc['유저D'].isna()].index))
print()
print('추천 결과 (예상 평점):')
recommend('유저D', ratings, user_sim_df)